---
title: Prior predictive interpretation
author: Bambi developers
---

Prior predictive checks explore a built model before fitting. The complete output
of `Model.prior_predictive()` contains latent `prior` draws as well as response
samples. `interpret` uses these latent draws to compute conditional predictions,
comparisons and slopes. A subset containing only `prior_predictive` responses
cannot reconstruct latent conditional effects.

We use proper explicit priors. With `auto_scale=False`, unspecified flat priors
cannot be prior-sampled. Automatic prior scaling, when enabled, uses the model
data even though it does not fit a likelihood. The `mean` target is the parent
parameter of the response family; it is not E[Y] for every family.


In [ ]:
import bambi as bmb
import numpy as np
import pandas as pd
from bambi.interpret import predictions, plot_predictions, comparisons, slopes

# Small synthetic covariates and responses; no external datasets or model.fit.
data = pd.DataFrame({
    "x": [-1., 0., 1., 2.] * 2,
    "group": ["a"] * 4 + ["b"] * 4,
    "y": [0.2, 0.5, 1.4, 1.7, 0.5, 0.8, 1.1, 2.1],
})
priors = {
    "Intercept": bmb.Prior("Normal", mu=0, sigma=1),
    "x": bmb.Prior("Normal", mu=0, sigma=0.5),
    "1|group": bmb.Prior("Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=0.5)),
    "sigma": bmb.Prior("HalfNormal", sigma=0.5),
}
model = bmb.Model("y ~ x + (1|group)", data, priors=priors,
                  auto_scale=False, center_predictors=False)
model.build()
prior = model.prior_predictive(draws=100, random_seed=819)
assert "posterior" not in prior.children

In [ ]:
# Remove group contributions for the population-level check.
population = predictions(
    model, prior, conditional={"x": [-1., 0., 1., 2.]},
    sample_group="prior", include_group_specific=False, prob=[0.5, 0.9],
)
population.summary

In [ ]:
# Include the sampled effects of existing groups on a conditional grid.
figure = plot_predictions(
    model, prior, conditional={"x": [-1., 0., 1., 2.], "group": ["a", "b"]},
    sample_group="prior", include_group_specific=True, random_seed=819,
)
figure

In [ ]:
# Apply a transformation to the response parameter before summarizing.
transformed = predictions(
    model, prior, conditional={"x": [-1., 0., 1., 2.]},
    sample_group="prior", include_group_specific=False,
    transforms={"y": np.exp},
)
contrast = comparisons(
    model, prior, contrast={"x": [0., 1.]}, sample_group="prior",
    include_group_specific=False,
)
slope = slopes(model, prior, wrt={"x": 0.}, sample_group="prior",
               include_group_specific=False)
assert "prior_predictions" in transformed.draws.children
assert np.isfinite(contrast.summary.estimate).all()
assert np.isfinite(slope.summary.estimate).all()
transformed.summary

Use `target=model.response_term.label` to include response noise, or a modeled
auxiliary parameter name to examine its prior distribution. Supply `random_seed`
for response sampling and new groups. If both posterior and prior draws are
present, the default selects posterior; use `sample_group="prior"` explicitly for
prior checks. Raw draws are retained in the returned result.

The direct `Model.predict` method also supports prior in-sample checks and draw
selection with `inplace=False`. For censored/truncated responses,
`kind="response_conditional"` still conditions on observed bounds; it is not an
unconditional prior simulation.

After changing priors with `set_priors`, call `build()` again and generate fresh
prior draws before repeating the checks. Nothing here fits or updates a likelihood.
